# VitiGuard MD — модель лозы: стволы и грозди

Ежедневное дообучение детектора «ствол + гроздь» (ml/pr-023, ml/pr-032):

- первый запуск собирает набор из открытых данных (INESC TEC и WGISD, см. `ml/DATASETS.md`) и сохраняет его на Диск;
- каждый запуск дообучает модель от последней принятой версии;
- новая версия принимается, только если на неизменной тестовой выборке она лучше прежней и ни один класс не просел больше чем на 1 пункт;
- веса, ONNX и реестр версий хранятся на Диске в `Мой диск/vitiguard/vine`.

**Перед запуском:** «Среда выполнения» → «Сменить среду выполнения» → T4 GPU, затем «Выполнить все». Запускайте раз в день.


In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv


In [ ]:
from google.colab import drive

drive.mount("/content/drive")
DRIVE = "/content/drive/MyDrive/vitiguard"
VINE = f"{DRIVE}/vine"
DATASET_ZIP = f"{VINE}/vine_dataset.zip"
# Ветка репозитория с кодом обучения
BRANCH = "main"
EPOCHS = 40


In [ ]:
!rm -rf /content/VitiGuard-MD_back
!git clone -q --depth 1 --branch {BRANCH} https://github.com/androtschii/VitiGuard-MD_back /content/VitiGuard-MD_back
%cd /content/VitiGuard-MD_back/ml
!pip install -q -e . ultralytics opencv-python-headless


## Набор данных

Первый раз — загрузка открытых наборов прямо в Colab (около 5 ГБ) и сборка; готовый набор сохраняется на Диск. В следующие разы он берётся с Диска, поэтому тестовая выборка всегда одна и та же. Если в коде появился новый источник, он дописывается в набор на Диске только в обучение и проверку.


In [ ]:
import os

os.makedirs(VINE, exist_ok=True)
!rm -rf /content/vine && mkdir -p /content/vine
if os.path.exists(DATASET_ZIP):
    !unzip -q "{DATASET_ZIP}" -d /content/vine
missing = !python -c "from pathlib import Path; from vitiguard_ml.data.vine import missing_sources; print(*missing_sources(Path('/content/vine')))"
missing = missing[0].split() if missing else []
if missing:
    print("Добавляются источники:", ", ".join(missing))
    if any(name.startswith("inesc") for name in missing):
        !mkdir -p data/raw/inesc
        !curl -fsSL -o data/raw/inesc/grape_trunk_dataset.zip "https://zenodo.org/records/5139598/files/grape_trunk_dataset.zip?download=1"
        !echo "9d2f80e3a44fe4ef046c41c08310673a  data/raw/inesc/grape_trunk_dataset.zip" | md5sum -c
    !python -m vitiguard_ml.data.vine --raw data/raw --out /content/vine
    !rm -f "{DATASET_ZIP}" && cd /content/vine && zip -qr "{DATASET_ZIP}" .
!cat /content/vine/sources.json


In [ ]:
# Пути в data.yaml — к набору в этой сессии Colab
!sed -i "s#^path: .*#path: /content/vine#" /content/vine/data.yaml


## Дообучение и сравнение с принятой версией


In [ ]:
!python -m vitiguard_ml.robot.vine --data /content/vine/data.yaml --out "{VINE}" --epochs {EPOCHS} --batch 16 --workers 2


## Реестр версий


In [ ]:
import json

with open(f"{VINE}/registry.json", encoding="utf-8") as file:
    registry = json.load(file)
print(f"{'версия':<8}{'дата':<22}{'mAP50-95':>9}{'ствол':>8}{'гроздь':>8}  принята")
for entry in registry:
    test = entry["test"]
    print(
        f"{entry['version']:<8}{entry['date'][:19]:<22}{test['map50_95']:>9.3f}"
        f"{test.get('trunk_map50_95', 0):>8.3f}{test.get('bunch_map50_95', 0):>8.3f}  "
        f"{'да' if entry['accepted'] else 'нет'}"
    )
